# ReLU (Rectified Linear Unit)

The default activation function for hidden layers in modern neural networks.
It replaced sigmoid in hidden layers because it's faster and avoids the vanishing gradient problem.

## 1. ReLU Function

**Formula:**

```
ReLU(x) = max(0, x)
```

- Negative input → `0` (neuron is "off")
- Positive input → pass through unchanged (neuron is "on")

**When to use:**
- Hidden layers of deep neural networks
- CNNs, feedforward networks, most modern architectures
- Default choice unless you have a specific reason to use something else

In [ ]:
def relu(x):
    return max(0, x)

test_values = [-10, -5, -1, 0, 1, 5, 10]

print("ReLU Function")
print("-" * 30)
print(f"{'Input':>8} | {'Output':>8}")
print("-" * 30)
for x in test_values:
    print(f"{x:>8} | {relu(x):>8}")

### ASCII visualization

In [ ]:
width = 50
height = 20
x_min, x_max = -6, 6
y_max = 6

print("ReLU Curve (x from -6 to 6)")
print("=" * (width + 10))

for row in range(height, -1, -1):
    y_level = y_max * row / height
    line = ""
    for col in range(width):
        x = x_min + (x_max - x_min) * col / (width - 1)
        y = relu(x)
        if abs(y - y_level) < 0.2:
            line += "*"
        elif col == width // 2:
            line += "|"
        elif row == 0:
            line += "-"
        else:
            line += " "
    label = f" {y_level:.0f}" if row % 5 == 0 else ""
    print(f"{line}{label}")

print(" " * (width // 2 - 3) + "-6    0    6")

## 2. ReLU Derivative

**Formula:**

```
ReLU'(x) = 0  if x < 0
ReLU'(x) = 1  if x > 0
```

The derivative is either 0 or 1 — a simple on/off switch.

**Why this matters:** Gradients don't shrink when flowing backward through ReLU layers.
Compare this to sigmoid's max derivative of 0.25.

In [ ]:
def relu_derivative(x):
    return 1 if x > 0 else 0

print("ReLU Derivative")
print("-" * 40)
print(f"{'Input':>8} | {'ReLU(x)':>8} | {'Derivative':>10}")
print("-" * 40)
for x in test_values:
    print(f"{x:>8} | {relu(x):>8} | {relu_derivative(x):>10}")

## 3. Why ReLU Replaced Sigmoid in Hidden Layers

In [ ]:
import math

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

# Gradient flow through 10 layers
# Best case: max derivative at each layer
sigmoid_max_deriv = 0.25
relu_max_deriv = 1.0

print("Gradient Flow: Sigmoid vs ReLU (best case)")
print("-" * 55)
print(f"{'Layers':>8} | {'Sigmoid gradient':>18} | {'ReLU gradient':>15}")
print("-" * 55)

sig_grad = 1.0
relu_grad = 1.0
for layer in range(1, 11):
    sig_grad *= sigmoid_max_deriv
    relu_grad *= relu_max_deriv
    print(f"{layer:>8} | {sig_grad:>18.10f} | {relu_grad:>15.1f}")

print(f"\nAfter 10 layers:")
print(f"  Sigmoid gradient: {sig_grad:.10f} (vanished!)")
print(f"  ReLU gradient:    {relu_grad:.1f} (preserved!)")

### Computation speed

In [ ]:
import time

n = 1_000_000
values = [i * 0.00001 - 5 for i in range(n)]

start = time.time()
for x in values:
    relu(x)
relu_time = time.time() - start

start = time.time()
for x in values:
    sigmoid(x)
sigmoid_time = time.time() - start

print(f"1,000,000 activations:")
print(f"  ReLU:    {relu_time:.3f}s")
print(f"  Sigmoid: {sigmoid_time:.3f}s")
print(f"  ReLU is {sigmoid_time / relu_time:.1f}x faster")
print(f"\nReLU is just max(0, x) — no expensive exp() needed.")

## 4. The Dying ReLU Problem

If a neuron always receives negative input, its output is always 0
and its gradient is always 0. It **never updates** — it's dead.

This can happen when:
- Learning rate is too high → weights get pushed to large negative values
- Bad weight initialization

In [ ]:
import random
random.seed(42)

# Simulate 10 neurons receiving random inputs
print("Simulating 10 neurons with random weights...")
print("(large negative bias = likely to die)")
print("-" * 55)

for i in range(10):
    bias = random.uniform(-5, 2)  # some neurons get unlucky
    weight = random.uniform(-1, 1)

    activations = 0
    total = 100
    for _ in range(total):
        x = random.uniform(-1, 1)
        output = relu(weight * x + bias)
        if output > 0:
            activations += 1

    pct = activations / total * 100
    if pct == 0:
        status = "DEAD"
    elif pct < 20:
        status = "barely alive"
    else:
        status = "healthy"

    print(f"  Neuron {i}: bias={bias:>+6.2f}  fired {activations:>3}/{total}  ({pct:>5.1f}%)  {status}")

## 5. ReLU Variants

Several variants fix the dying ReLU problem by allowing a small gradient for negative inputs.

In [ ]:
def leaky_relu(x, alpha=0.01):
    return x if x > 0 else alpha * x

def elu(x, alpha=1.0):
    return x if x > 0 else alpha * (math.exp(x) - 1)

test = [-5, -2, -1, -0.5, 0, 0.5, 1, 2, 5]

print("ReLU vs Leaky ReLU vs ELU")
print("-" * 60)
print(f"{'x':>6} | {'ReLU':>8} | {'Leaky ReLU':>11} | {'ELU':>8}")
print("-" * 60)
for x in test:
    print(f"{x:>6.1f} | {relu(x):>8.4f} | {leaky_relu(x):>11.4f} | {elu(x):>8.4f}")

print("\nKey difference for negative inputs:")
print("  ReLU:       always 0 (dead zone)")
print("  Leaky ReLU: small slope (0.01x) — gradient survives")
print("  ELU:        smooth curve — approaches -1 asymptotically")

### ASCII comparison: ReLU vs Leaky ReLU

In [ ]:
width = 50
height = 20
x_min, x_max = -6, 6
y_min, y_max = -1, 6

print("R = ReLU    L = Leaky ReLU (alpha=0.1 for visibility)")
print("=" * (width + 10))

for row in range(height, -1, -1):
    y_level = y_min + (y_max - y_min) * row / height
    line = ""
    for col in range(width):
        x = x_min + (x_max - x_min) * col / (width - 1)
        y_relu = relu(x)
        y_leaky = leaky_relu(x, alpha=0.1)
        on_relu = abs(y_relu - y_level) < 0.2
        on_leaky = abs(y_leaky - y_level) < 0.2
        if on_relu and on_leaky:
            line += "X"
        elif on_relu:
            line += "R"
        elif on_leaky:
            line += "L"
        elif abs(y_level) < 0.15:
            line += "-"
        elif col == width // 2:
            line += "|"
        else:
            line += " "
    label = f" {y_level:.0f}" if row % (height // 5) == 0 else ""
    print(f"{line}{label}")

print(f"\n  R = ReLU (flat at 0)    L = Leaky ReLU (small slope)    X = overlap")

## 6. NumPy Version (Vectorized)

In [ ]:
import numpy as np

def relu_np(x):
    return np.maximum(0, x)

def relu_derivative_np(x):
    return (x > 0).astype(float)

def leaky_relu_np(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

x = np.array([-5, -2, -1, 0, 1, 2, 5])

print("NumPy vectorized ReLU")
print(f"Input:        {x}")
print(f"ReLU:         {relu_np(x)}")
print(f"Derivative:   {relu_derivative_np(x)}")
print(f"Leaky ReLU:   {leaky_relu_np(x)}")

## Summary

| | ReLU | Sigmoid |
|---|---|---|
| **Formula** | `max(0, x)` | `1 / (1 + e^(-x))` |
| **Output range** | [0, ∞) | (0, 1) |
| **Derivative** | 0 or 1 | 0 to 0.25 |
| **Speed** | Very fast | Slow (exp) |
| **Vanishing gradient** | No | Yes |
| **Weakness** | Dying neurons | Vanishing gradients |
| **Best for** | Hidden layers | Output layer (binary) |

### Which to use where

- **Hidden layers** → ReLU (or Leaky ReLU)
- **Binary output** → Sigmoid (probability between 0 and 1)
- **Multi-class output** → Softmax (probabilities that sum to 1)